# Наши прогоны: маскирование голов на Qwen1.5-14B-Chat

Визуализация по образцу `CreateVizFromLLMTesting.ipynb` авторов, для нескольких прогонов сразу.

Читает `results/graph/<папка>/*_results.json`. Прогоны, которых ещё нет, пропускаются.

In [ ]:
import glob
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

## Какие прогоны показывать

Подпись → папка в `results/graph/`. Чтобы добавить прогон (например, top-50), допишите строку.

In [ ]:
model_name = 'Qwen1.5-14B-Chat'
runs = {
    'без маски': model_name,
    'top-30': f'{model_name}_block_top30',
    'random-30': f'{model_name}_block_random30',
}
# цвет закреплён за прогоном, а не за порядком на графике
colors = {'без маски': '#2a78d6', 'top-30': '#eb6834', 'random-30': '#1baf7a'}

# ноутбук может лежать в source/viz или запускаться из source
graph_dir = '../results/graph' if os.path.isdir('../results/graph') else 'results/graph'
out_dir = 'viz' if os.path.isdir('viz') else '.'

## Загрузка

In [ ]:
data = []
for run, folder in runs.items():
    files = glob.glob(f'{graph_dir}/{folder}/*_results.json')
    if not files:
        print(f'нет результатов: {run} ({graph_dir}/{folder})')
        continue
    for file in files:
        with open(file, 'r') as f:
            json_data = json.load(f)
        data.append({
            'Run': run,
            'Document Depth': json_data.get('depth_percent', None),
            'Context Length': json_data.get('context_length', None),
            'Score': json_data.get('score', None),
        })

df = pd.DataFrame(data)
found = [run for run in runs if run in set(df['Run'])]
print(f'прогонов: {len(found)}, строк: {len(df)}')

## Сводка

`Успешных, %` — доля примеров с ROUGE-1 recall > 50, тот же порог, что в коде авторов.

In [ ]:
summary = df.groupby('Run')['Score'].agg(
    **{'Примеров': 'count', 'Средний ROUGE': 'mean', 'Успешных, %': lambda s: (s > 50).mean() * 100}
).loc[found].round(1)
summary

## Качество по длине контекста

Среднее по всем глубинам на каждой длине.

In [ ]:
by_len = df.pivot_table(values='Score', index='Context Length', columns='Run', aggfunc='mean')[found]

fig, ax = plt.subplots(figsize=(11, 5))
for run in found:
    ax.plot(by_len.index, by_len[run], color=colors.get(run), linewidth=2, marker='o', markersize=5, label=run)
ax.set_ylim(0, 105)
ax.set_xlabel('Длина контекста, токенов')
ax.set_ylabel('Средний ROUGE-1 recall')
ax.set_title(f'{model_name}: качество поиска иглы при маскировании голов')
ax.grid(axis='y', color='#e6e5e1', linewidth=1)
ax.set_axisbelow(True)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, loc='lower left')
fig.tight_layout()
fig.savefig(f'{out_dir}/ours_by_length.png', dpi=150)
plt.show()

by_len.round(1)

## Тепловые карты, как у авторов

Цвета и шкала те же, что в `top30.png` и `random.png` авторов, чтобы картинки можно было сравнивать.

Ось глубины читать осторожно: в этих прогонах игла на Qwen вставляется не на заданной глубине (см. PLAN.md, раздел 3.4). Глубина 100 — это конец контекста, остальные строки — начало или случайное место перед заданной точкой.

In [ ]:
cmap = LinearSegmentedColormap.from_list('custom_cmap', ['#F0496E', '#EBB839', '#0CD79F'])

fig, axes = plt.subplots(len(found), 1, figsize=(17.5, 5.5 * len(found)), squeeze=False)
for ax, run in zip(axes[:, 0], found):
    pivot_table = df[df['Run'] == run].pivot_table(values='Score', index='Document Depth', columns='Context Length', aggfunc='mean')
    sns.heatmap(pivot_table, fmt='g', cmap=cmap, cbar_kws={'label': 'Score'}, vmin=0, vmax=100, ax=ax)
    ax.set_title(f'{model_name}, {run}: средний ROUGE {df[df["Run"] == run]["Score"].mean():.1f}')
    ax.set_xlabel('Token Limit')
    ax.set_ylabel('Depth Percent')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
fig.tight_layout()
fig.savefig(f'{out_dir}/ours_heatmaps.png', dpi=150)
plt.show()

## Попарное сравнение

Одни и те же 200 пар (длина, глубина) в каждом прогоне. `Разница` — на сколько пунктов ROUGE первый прогон ниже второго в среднем; `Хуже` и `Лучше` — на скольких примерах.

In [ ]:
wide = df.pivot_table(values='Score', index=['Context Length', 'Document Depth'], columns='Run', aggfunc='mean')

rows = []
for a, b in [('top-30', 'без маски'), ('random-30', 'без маски'), ('top-30', 'random-30')]:
    if a in found and b in found:
        pair = wide[[a, b]].dropna()
        diff = pair[a] - pair[b]
        rows.append({'Сравнение': f'{a} против {b}', 'Примеров': len(pair), 'Разница': round(diff.mean(), 1),
                     'Хуже': int((diff < 0).sum()), 'Лучше': int((diff > 0).sum()), 'Без изменений': int((diff == 0).sum())})
pd.DataFrame(rows)